# Setup

In [1]:
import os
import sys

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("."), "src")))
import pandas as pd
from processor.main import Processor
from processor.table.representation.impl.df_table import DFTable
from processor.table.store.table_store_factory import ImplementedTableStore

In [2]:
# OpenAI model
# from dotenv import load_dotenv
# load_dotenv()
# api_key = os.getenv('OPENAI_API_KEY')
# model = 'gpt-4o-mini-2024-07-18'

# Local Model
model = "src/processor/model/weight/qwen25-7b"
embed_path = "src/processor/model/weight/bge-base"
processor = Processor(
    model, embed_path, ImplementedTableStore.PY_TABLE_STORE, "processor_output/tariff"
)

In [3]:
# Define constants
DB_SCHEMA = "E2E_SCHEMA"
BEFORE_UNION_DB_SCHEMA = "BEFORE_UNION_DB_SCHEMA"
DB_SCHEMA_AFTER_UNION = DB_SCHEMA + "_AFTER_UNION"
DB_SCHEMA_FINAL = "db_schema_final"
QUESTION = 'A supplier from Germany, associated with PO ID 40411289, has just announced the application of minimum import tariffs set by the U.S. government on goods imported from Germany. Previously, there were no such tariffs. How much of a difference will this make in terms of monetary value (in USD)?'

In [4]:
# Prepare metadata (table descriptions)
metadata = pd.read_csv("data_src/buysite/metadata.csv")
table_descriptions: dict[str, str] = dict()
for i, row in metadata.iterrows():
    table_descriptions[row["table"]] = row["value"]

## Skip if already indexed

In [5]:
# # Index the tables, retrieved by Pneuma for QUESTION_1
# from tqdm import tqdm
# import duckdb

# processor.ctx.table_store.create_db_schema(DB_SCHEMA)
# TABLE_PATH_PREFIX = "data_src/buysite/dataset"
# retrieved_tables = [
#     "JI_PURCHASE_ORDER_AUDIT_TRAIL",
#     "JI_REQUISITION_AUDIT_TRAIL",
#     "JI_PURCHASE_ORDER_LINE",
#     "JI_PURCHASE_ORDER_CUSTOM_FIELDS_SINGLE_VALUE_RESPONSE",
#     "JI_PURCHASE_ORDER",
# ]
# for table_name in tqdm(retrieved_tables):
#     table_path = os.path.join(TABLE_PATH_PREFIX, f"{table_name}.csv")
#     query = f"""
#     SELECT * FROM read_csv_auto('{table_path}')
#     LIMIT 1000"""
#     df = duckdb.query(query).to_df()
#     processor.ctx.table_store.add_table(
#         DB_SCHEMA,
#         table_name,
#         DFTable(df),
#         False,
#         False,
#     )
# processor.ctx.table_store.checkpoint()

# Step 1: Schema Enhancement & Target Schema Generation

## 1a. Produce Target Schema

> 2m45s

In [5]:
target_schema_node = processor.get_target_schema(QUESTION)
target_schema = target_schema_node.computation_output
print(target_schema)

[2025-05-06 04:24:00] INFO in schema_processor: Getting target schema for the question A supplier from Germany, associated with PO ID 40411289, has just announced the application of minimum import tariffs set by the U.S. government on goods imported from Germany. Previously, there were no such tariffs. How much of a difference will this make in terms of monetary value (in USD)?


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[2025-05-06 04:25:09] INFO in schema_processor: => Target schema: {
    "schema": {
        "ID": {
            "description": "Unique identifier for each record",
            "type": "INTEGER"
        },
        "PO_ID": {
            "description": "Purchase Order ID",
            "type": "VARCHAR"
        },
        "Supplier": {
            "description": "Name of the supplier",
            "type": "VARCHAR"
        },
        "Country_of_Supplier": {
            "description": "Country where the supplier is located",
            "type": "VARCHAR"
        },
        "Import_Tariff": {
            "description": "Import tariff percentage applied to goods from the supplier",
            "type": "FLOAT"
        },
        "Monetary_Value_USD": {
            "description": "Monetary value of the order in USD",
            "type": "FLOAT"
        }
    },
    "sql_query": "SELECT (Import_Tariff / 100 * Monetary_Value_USD) AS Tariff_Difference FROM target_table WHERE PO_ID = 40411289 AND

In [7]:
output = [
    {
        "schema": {
            "PO_ID": {"description": "Purchase Order ID", "type": "VARCHAR"},
            "Supplier": {"description": "Name of the supplier", "type": "VARCHAR"},
            "Country_of_Supplier": {
                "description": "Country where the supplier is located",
                "type": "VARCHAR",
            },
            "Import_Tariff": {
                "description": "Import tariff percentage applied to goods from the supplier",
                "type": "FLOAT",
            },
            "Monetary_Value_USD": {
                "description": "Monetary value of the order in USD",
                "type": "FLOAT",
            },
        },
        "reasoning": "The PO_ID uniquely identifies a row, so all other filters (such as Supplier and Country_of_Supplier) are redundant. The Import_Tariff is not null as per the schema and the question, so the check for `IS NOT NULL` is unnecessary. The query can be simplified to only use the PO_ID for filtering.",
        "sql_query": "SELECT (Import_Tariff / 100 * Monetary_Value_USD) AS Tariff_Difference FROM target_table WHERE PO_ID = 40411289",
    }
]
target_schema = output[0]["schema"]
sql_query = output[0]["sql_query"]

## 1b. Enhance Table Schemas

> 8.5 minutes

In [56]:
table_descriptions_node = processor.get_table_descriptions(DB_SCHEMA, existing_descriptions=table_descriptions)
table_descriptions = table_descriptions_node.computation_output
print(table_descriptions)

Describing tables:   0%|          | 0/5 [00:00<?, ?it/s]

[2025-05-05 05:51:46] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-05-05 05:53:05] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-05-05 05:53:19] INFO in schema_processor: => Overall description of table 'JI_PURCHASE_ORDER_AUDIT_TRAIL': This table captures detailed information about actions taken on purchase orders (POs) within an organization, including the date and time of the action, the user who performed it, the type of action, affected fields, old and new values, import status, and additional notes.


Describing tables:  20%|██        | 1/5 [01:32<06:10, 92.72s/it]

[2025-05-05 05:53:19] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-05-05 05:54:19] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-05-05 05:54:26] INFO in schema_processor: => Overall description of table 'JI_REQUISITION_AUDIT_TRAIL': This table captures detailed information on various actions taken during the lifecycle of requisitions, including timestamps, user identifiers, and field changes.


Describing tables:  40%|████      | 2/5 [02:39<03:52, 77.60s/it]

...

[2025-05-05 05:56:35] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-05-05 05:58:24] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-05-05 05:58:44] INFO in schema_processor: => Overall description of table 'JI_PURCHASE_ORDER_CUSTOM_FIELDS_SINGLE_VALUE_RESPONSE': This table contains detailed information about purchase orders, including various custom fields specific to each organization. Each row represents a unique purchase order line item, identified by `PO_ID`, `ORG_ID`, and `PO_LINE_ID`. Custom fields prefixed with `CF_` store additional metadata relevant to the purchase order, and the `ELT_TS` column indicates when the data was last processed or loaded.


Describing tables:  80%|████████  | 4/5 [06:57<01:52, 112.05s/it]

[2025-05-05 05:58:44] INFO in schema_processor: Step 1: Sample rows multiple times to get different perspectives.
[2025-05-05 05:59:58] INFO in schema_processor: Step 2: Combine all perspectives.
[2025-05-05 06:00:10] INFO in schema_processor: => Overall description of table 'JI_PURCHASE_ORDER': This table provides comprehensive details on purchase orders, including metadata such as organization ID, purchase order number, and supplier information, along with financial details like total amounts in different currencies and exchange rates, and workflow statuses indicating the current state of the purchase order process.


Describing tables: 100%|██████████| 5/5 [08:24<00:00, 100.84s/it]

{'JI_PURCHASE_ORDER_AUDIT_TRAIL': 'This table captures detailed information about actions taken on purchase orders (POs) within an organization, including the date and time of the action, the user who performed it, the type of action, affected fields, old and new values, import status, and additional notes.', 'JI_REQUISITION_AUDIT_TRAIL': 'This table captures detailed information on various actions taken during the lifecycle of requisitions, including timestamps, user identifiers, and field changes.', 'JI_PURCHASE_ORDER_LINE': 'This table provides comprehensive details on purchase order (PO) lines, including information on the organization, purchase order, line item, supplier, and financial aspects. It captures metadata such as creation and distribution timestamps, accounting dates, and statuses related to receipts, invoices, and shipments. The table also includes pricing details in different currencies and unit prices, as well as classification information for the items ordered. Addit

In [ ]:
table_descriptions = {
    "JI_PURCHASE_ORDER_AUDIT_TRAIL": "This table captures detailed information about actions taken on purchase orders (POs) within an organization, including the date and time of the action, the user who performed it, the type of action, affected fields, old and new values, import status, and additional notes.",
    "JI_REQUISITION_AUDIT_TRAIL": "This table captures detailed information on various actions taken during the lifecycle of requisitions, including timestamps, user identifiers, and field changes.",
    "JI_PURCHASE_ORDER_LINE": "This table provides comprehensive details on purchase order (PO) lines, including information on the organization, purchase order, line item, supplier, and financial aspects. It captures metadata such as creation and distribution timestamps, accounting dates, and statuses related to receipts, invoices, and shipments. The table also includes pricing details in different currencies and unit prices, as well as classification information for the items ordered. Additionally, it tracks the workflow status and any special conditions or actions associated with each PO line.",
    "JI_PURCHASE_ORDER_CUSTOM_FIELDS_SINGLE_VALUE_RESPONSE": "This table contains detailed information about purchase orders, including various custom fields specific to each organization. Each row represents a unique purchase order line item, identified by `PO_ID`, `ORG_ID`, and `PO_LINE_ID`. Custom fields prefixed with `CF_` store additional metadata relevant to the purchase order, and the `ELT_TS` column indicates when the data was last processed or loaded.",
    "JI_PURCHASE_ORDER": "This table provides comprehensive details on purchase orders, including metadata such as organization ID, purchase order number, and supplier information, along with financial details like total amounts in different currencies and exchange rates, and workflow statuses indicating the current state of the purchase order process.",
}

# Step 2: Base Table Producer

## 2a. Define Constants from the Previous Step

In [8]:
from tqdm import tqdm

In [6]:
target_schema = {
            "PO_ID": {"description": "Purchase Order ID", "type": "VARCHAR"},
            "Supplier": {"description": "Name of the supplier", "type": "VARCHAR"},
            "Country_of_Supplier": {
                "description": "Country where the supplier is located",
                "type": "VARCHAR",
            },
            "Import_Tariff": {
                "description": "Import tariff percentage applied to goods from the supplier",
                "type": "FLOAT",
            },
            "Monetary_Value_USD": {
                "description": "Monetary value of the order in USD",
                "type": "FLOAT",
            },
        }
sql_query = "SELECT (Import_Tariff / 100 * Monetary_Value_USD) AS Tariff_Difference FROM target_table WHERE PO_ID = 40411289"
table_descriptions = {
    "JI_PURCHASE_ORDER_AUDIT_TRAIL": "This table captures detailed information about actions taken on purchase orders (POs) within an organization, including the date and time of the action, the user who performed it, the type of action, affected fields, old and new values, import status, and additional notes.",
    "JI_REQUISITION_AUDIT_TRAIL": "This table captures detailed information on various actions taken during the lifecycle of requisitions, including timestamps, user identifiers, and field changes.",
    "JI_PURCHASE_ORDER_LINE": "This table provides comprehensive details on purchase order (PO) lines, including information on the organization, purchase order, line item, supplier, and financial aspects. It captures metadata such as creation and distribution timestamps, accounting dates, and statuses related to receipts, invoices, and shipments. The table also includes pricing details in different currencies and unit prices, as well as classification information for the items ordered. Additionally, it tracks the workflow status and any special conditions or actions associated with each PO line.",
    "JI_PURCHASE_ORDER_CUSTOM_FIELDS_SINGLE_VALUE_RESPONSE": "This table contains detailed information about purchase orders, including various custom fields specific to each organization. Each row represents a unique purchase order line item, identified by `PO_ID`, `ORG_ID`, and `PO_LINE_ID`. Custom fields prefixed with `CF_` store additional metadata relevant to the purchase order, and the `ELT_TS` column indicates when the data was last processed or loaded.",
    "JI_PURCHASE_ORDER": "This table provides comprehensive details on purchase orders, including metadata such as organization ID, purchase order number, and supplier information, along with financial details like total amounts in different currencies and exchange rates, and workflow statuses indicating the current state of the purchase order process.",
}

## 2b. Relevant Table Selection

> 4m10s

In [7]:
relevant_table_ids_node = processor.select_relevant_table_ids(
    DB_SCHEMA,
    sql_query,
    target_schema,
    table_descriptions,
    # 3,
    # [target_schema_node, table_descriptions_node],
)
relevant_table_ids = relevant_table_ids_node.computation_output
print(relevant_table_ids)

[2025-05-06 05:12:02] INFO in base_table_producer: Checking the relevance of table `JI_PURCHASE_ORDER_AUDIT_TRAIL`


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[2025-05-06 05:12:40] INFO in base_table_producer: => Table relevancy output: Relevant: no

Reasoning:
The provided table contains detailed information about actions taken on purchase orders, such as dates, users, action codes, and field changes. However, it does not contain any columns related to the supplier's name, country, import tariff, or monetary value of the order in USD. The SQL query requires specific fields like `Import_Tariff` and `Monetary_Value_USD`, which are not present in the given table. Therefore, this table does not contribute any useful information towards fulfilling the target schema or the SQL query intent.
[2025-05-06 05:12:40] INFO in base_table_producer: Checking the relevance of table `JI_REQUISITION_AUDIT_TRAIL`
[2025-05-06 05:13:10] INFO in base_table_producer: => Table relevancy output: Relevant: no

Reasoning: The provided table contains information related to actions taken on requisitions, such as timestamps, user identifiers, and field changes. However,

## 2c. Table Union

### Produce Operations

In [48]:
relevant_table_ids = ['JI_PURCHASE_ORDER_LINE', 'JI_PURCHASE_ORDER']

In [ ]:
try:
    processor.ctx.table_store.create_db_schema(BEFORE_UNION_DB_SCHEMA)
except:
    pass
for table_name in tqdm(relevant_table_ids):
    table = processor.ctx.table_store.get_table(DB_SCHEMA, table_name)
    processor.ctx.table_store.add_table(
        BEFORE_UNION_DB_SCHEMA, table_name, table, True
    )
processor.ctx.table_store.checkpoint()

100%|██████████| 2/2 [00:00<00:00, 22250.95it/s]


> 14m6s

In [7]:
union_operations_node = processor.produce_union_operations(
    BEFORE_UNION_DB_SCHEMA,
    table_descriptions,
    3,
    # [target_schema_node, table_descriptions_node],
)
union_operations = union_operations_node.computation_output
print(union_operations)

[2025-05-06 05:55:15] INFO in base_table_producer: => available_tables_formatted: - JI_PURCHASE_ORDER_LINE (Provides details on the purchase order lines.):
```col: ORG_ID | PO_ID | PO_LINE_ID | DEPT_KEY | SUPPLIER_KEY | ITEM_KEY | PO_NUMBER | CONTRACT_ID | CONTRACT_NUMBER | QUANTITY | EXTENDED_PRICE | CREATED_TS | DISTRIBUTION_TS | EXPORT_TS | LASTREVISION_TS | ORIGINALREVISION_TS | WORKFLOWCOMPLETED_TS | ACCOUNTING_DATE | USER_OWNER_KEY | USER_SUBMITTER_KEY | EXTERNAL_PO_ID | LINE_NUMBER | UNIT_PRICE | CONTRACT_UNIT_PRICE | SUPPLIER_ACCOUNT_CODE | SHIPPING_METHOD | IS_PO_LINE_AWARDED_BID | IS_PO_LINE_REJECTED | IS_PO_LINE_CANCELLED | IS_PO_LINE_SENT_TO_SUPPLIER | IS_PO_LINE_HAS_INVOICES | IS_PO_LINE_FORCE_MATCHED | IS_PO_FORCE_MATCHED | REQUISITION_ID | REQUISITION_NAME | REQUISITION_LINE_ID | REQUISITION_LINE_NUMBER | REQUISITION_CREATED_TS | PO_LAST_REVISION_NUMBER | IS_PO_HAS_REJECTED_ITEMS | IS_PO_HAS_CREDITS | IS_PO_OVERSHIPPED | IS_PO_OVERRECEIVED | IS_PO_OVERINVOICED | IS_PO_HA

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

[2025-05-06 05:56:31] INFO in base_table_producer: => reasoning: ### Analysis

The `JI_PURCHASE_ORDER_LINE` table contains detailed information about individual lines within a purchase order, including quantities, prices, and various statuses related to the line items.

The `JI_PURCHASE_ORDER` table provides summary information about the entire purchase order, such as the total amount, receipt status, invoice status, and workflow status.

### Row Extension Groups

#### Group 1: `JI_PURCHASE_ORDER_LINE`, `JI_PURCHASE_ORDER`

- **Reasoning**: Both tables are related to purchase orders. The `JI_PURCHASE_ORDER_LINE` table contains detailed information for each line item in a purchase order, while the `JI_PURCHASE_ORDER` table provides a summary of the entire purchase order. These tables can be combined by matching the `ORG_ID` and `PO_ID` columns, which uniquely identify each purchase order. The `PO_ID` in `JI_PURCHASE_ORDER_LINE` corresponds to the `PO_ID` in `JI_PURCHASE_ORDER`.

### Fin

KeyboardInterrupt: 

### Execute Operations

In [49]:
union_operations = [
    {'Output Table ID': 'Union_1', 'Tables': ['JI_PURCHASE_ORDER_LINE', 'JI_PURCHASE_ORDER'], 'Unified Schema': ['ORG_ID', 'PO_ID', 'PO_LINE_ID', 'DEPT_KEY', 'SUPPLIER_KEY', 'ITEM_KEY', 'PO_NUMBER', 'CONTRACT_ID', 'CONTRACT_NUMBER', 'QUANTITY', 'EXTENDED_PRICE', 'CREATED_TS', 'DISTRIBUTION_TS', 'EXPORT_TS', 'LASTREVISION_TS', 'ORIGINALREVISION_TS', 'WORKFLOWCOMPLETED_TS', 'ACCOUNTING_DATE', 'USER_OWNER_KEY', 'USER_SUBMITTER_KEY', 'EXTERNAL_PO_ID', 'LINE_NUMBER', 'UNIT_PRICE', 'CONTRACT_UNIT_PRICE', 'SUPPLIER_ACCOUNT_CODE', 'SHIPPING_METHOD', 'IS_PO_LINE_AWARDED_BID', 'IS_PO_LINE_REJECTED', 'IS_PO_LINE_CANCELLED', 'IS_PO_LINE_SENT_TO_SUPPLIER', 'IS_PO_LINE_HAS_INVOICES', 'IS_PO_LINE_FORCE_MATCHED', 'IS_PO_FORCE_MATCHED', 'REQUISITION_ID', 'REQUISITION_NAME', 'REQUISITION_LINE_ID', 'REQUISITION_LINE_NUMBER', 'REQUISITION_CREATED_TS', 'PO_LAST_REVISION_NUMBER', 'IS_PO_HAS_REJECTED_ITEMS', 'IS_PO_HAS_CREDITS', 'IS_PO_OVERSHIPPED', 'IS_PO_OVERRECEIVED', 'IS_PO_OVERINVOICED', 'IS_PO_HAS_SUBSTITUTED_INVOICE_ITEMS', 'IS_PO_HAS_CANCELLED_RECEIPT_ITEMS', 'IS_PO_HAS_CANCELLED_ITEMS', 'IS_PO_HAS_RECEIPTS', 'IS_PO_HAS_RETURNS', 'IS_PO_HAS_INVOICES', 'UNIT_PRICE_USD', 'EXTENDED_PRICE_USD', 'CONTRACT_UNIT_PRICE_USD', 'SUPPLIER_RANK', 'DIVERSIFIED_SUPPLIER', 'LIST_PRICE_SET_ID', 'LIST_PRICE_SET_NAME', 'LIST_PRICE', 'LIST_PRICE_SET_VERSION', 'PREVIOUS_LIST_PRICE', 'PREVIOUS_LIST_PRICE_SET_VERSION', 'PO_TYPE_ENUM', 'PO_TYPE', 'PO_RECEIPT_STATUS_ENUM', 'PO_RECEIPT_STATUS', 'PO_INVOICE_STATUS_ENUM', 'PO_INVOICE_STATUS', 'PO_WORKFLOW_STATUS_ENUM', 'PO_WORKFLOW_STATUS', 'PO_MATCH_STATUS_ENUM', 'PO_MATCH_STATUS', 'UNIT_PRICE_SOURCE_ENUM', 'UNIT_PRICE_SOURCE', 'PO_LINE_MATCH_STATUS_ENUM', 'PO_LINE_MATCH_STATUS', 'CONTRACT_UNIT_PRICE_BUSINESS', 'CONTRACT_UNIT_PRICE_BUSINESS_CURRENCY', 'CONTRACT_UNIT_PRICE_BUSINESS_EXCHANGE_RATE', 'EXTENDED_PRICE_BUSINESS', 'EXTENDED_PRICE_BUSINESS_CURRENCY', 'EXTENDED_PRICE_BUSINESS_EXCHANGE_RATE', 'UNIT_PRICE_BUSINESS', 'UNIT_PRICE_BUSINESS_CURRENCY', 'UNIT_PRICE_BUSINESS_EXCHANGE_RATE', 'IS_PO_LINE_HAS_SHIPPED_ITEMS', 'IS_PO_LINE_HAS_RECEIPTS', 'PO_LINE_RECEIPT_STATUS_ENUM', 'PO_LINE_RECEIPT_STATUS', 'PO_LINE_SHIPMENT_STATUS_ENUM', 'PO_LINE_SHIPMENT_STATUS', 'RECEIPT_MAX_UNIT_PRICE', 'RECEIPT_MIN_UNIT_PRICE', 'IS_PO_LINE_WITH_CANCELLED_RECEIPT_ITEMS', 'IS_PO_LINE_MATCHING_NEEDS_RECEIPT', 'IS_PO_LINE_MATCHING_ONLY_NEEDS_RECEIPT', 'IS_PO_LINE_OVERSHIPPED', 'RECEIPT_MAX_UNIT_PRICE_USD', 'RECEIPT_MIN_UNIT_PRICE_USD', 'SHIP_TO_ADDRESS_ID', 'BILL_TO_ADDRESS_ID', 'COMMODITY_CODE_KEY', 'REQUESTED_DELIVERY_DATE', 'DELIVERY_DATE_TYPE', 'DELIVERY_LEAD_TIME_DAYS', 'FORM_ID', 'FORMREQUEST_ID', 'GRAND_TOTAL_AMOUNT', 'GRAND_TOTAL_DOCUMENT_AMOUNT', 'GRAND_TOTAL_DOCUMENT_CURRENCY', 'GRAND_TOTAL_DOCUMENT_EXCHANGE_RATE', 'GRAND_TOTAL_USD', 'ELT_TS', 'FULFILLMENT_CENTER_KEY', 'CATEGORY_LEVEL_1_NAME', 'CATEGORY_LEVEL_1_UNSPSC', 'CATEGORY_LEVEL_2_NAME', 'CATEGORY_LEVEL_2_UNSPSC', 'ORG_ID', 'PO_ID', 'DEPT_KEY', 'PO_NUMBER', 'EXTERNAL_PO_ID', 'PO_LAST_REVISION_NUMBER', 'IS_PO_HAS_REJECTED_ITEMS', 'IS_PO_HAS_CREDITS', 'IS_PO_OVERSHIPPED', 'IS_PO_OVERRECEIVED', 'IS_PO_OVERINVOICED', 'IS_PO_HAS_SUBSTITUTED_INVOICE_ITEMS', 'IS_PO_HAS_CANCELLED_RECEIPT_ITEMS', 'IS_PO_HAS_CANCELLED_ITEMS', 'IS_PO_HAS_RECEIPTS', 'IS_PO_HAS_RETURNS', 'IS_PO_HAS_INVOICES', 'GRAND_TOTAL', 'GRAND_TOTAL_USD', 'PO_TYPE_ENUM', 'PO_TYPE', 'PO_RECEIPT_STATUS_ENUM', 'PO_RECEIPT_STATUS', 'PO_INVOICE_STATUS_ENUM', 'PO_INVOICE_STATUS', 'PO_WORKFLOW_STATUS_ENUM', 'PO_WORKFLOW_STATUS', 'PO_AP_STATUS_ENUM', 'PO_AP_STATUS', 'PO_MATCH_STATUS_ENUM', 'PO_MATCH_STATUS', 'GRAND_TOTAL_BUSINESS', 'GRAND_TOTAL_BUSINESS_CURRENCY', 'GRAND_TOTAL_BUSINESS_EXCHANGE_RATE', 'ELT_TS'], 'Mappings': {'JI_PURCHASE_ORDER_LINE': {'ORG_ID': 'ORG_ID', 'PO_ID': 'PO_ID', 'PO_LINE_ID': 'PO_LINE_ID', 'DEPT_KEY': 'DEPT_KEY', 'SUPPLIER_KEY': 'SUPPLIER_KEY', 'ITEM_KEY': 'ITEM_KEY', 'PO_NUMBER': 'PO_NUMBER', 'CONTRACT_ID': 'CONTRACT_ID', 'CONTRACT_NUMBER': 'CONTRACT_NUMBER', 'QUANTITY': 'QUANTITY', 'EXTENDED_PRICE': 'EXTENDED_PRICE', 'CREATED_TS': 'CREATED_TS', 'DISTRIBUTION_TS': 'DISTRIBUTION_TS', 'EXPORT_TS': 'EXPORT_TS', 'LASTREVISION_TS': 'LASTREVISION_TS', 'ORIGINALREVISION_TS': 'ORIGINALREVISION_TS', 'WORKFLOWCOMPLETED_TS': 'WORKFLOWCOMPLETED_TS', 'ACCOUNTING_DATE': 'ACCOUNTING_DATE', 'USER_OWNER_KEY': 'USER_OWNER_KEY', 'USER_SUBMITTER_KEY': 'USER_SUBMITTER_KEY', 'EXTERNAL_PO_ID': 'EXTERNAL_PO_ID', 'LINE_NUMBER': 'LINE_NUMBER', 'UNIT_PRICE': 'UNIT_PRICE', 'CONTRACT_UNIT_PRICE': 'CONTRACT_UNIT_PRICE', 'SUPPLIER_ACCOUNT_CODE': 'SUPPLIER_ACCOUNT_CODE', 'SHIPPING_METHOD': 'SHIPPING_METHOD', 'IS_PO_LINE_AWARDED_BID': 'IS_PO_LINE_AWARDED_BID', 'IS_PO_LINE_REJECTED': 'IS_PO_LINE_REJECTED', 'IS_PO_LINE_CANCELLED': 'IS_PO_LINE_CANCELLED', 'IS_PO_LINE_SENT_TO_SUPPLIER': 'IS_PO_LINE_SENT_TO_SUPPLIER', 'IS_PO_LINE_HAS_INVOICES': 'IS_PO_LINE_HAS_INVOICES', 'IS_PO_LINE_FORCE_MATCHED': 'IS_PO_LINE_FORCE_MATCHED', 'IS_PO_FORCE_MATCHED': 'IS_PO_FORCE_MATCHED', 'REQUISITION_ID': 'REQUISITION_ID', 'REQUISITION_NAME': 'REQUISITION_NAME', 'REQUISITION_LINE_ID': 'REQUISITION_LINE_ID', 'REQUISITION_LINE_NUMBER': 'REQUISITION_LINE_NUMBER', 'REQUISITION_CREATED_TS': 'REQUISITION_CREATED_TS', 'PO_LAST_REVISION_NUMBER': 'PO_LAST_REVISION_NUMBER', 'IS_PO_HAS_REJECTED_ITEMS': 'IS_PO_HAS_REJECTED_ITEMS', 'IS_PO_HAS_CREDITS': 'IS_PO_HAS_CREDITS', 'IS_PO_OVERSHIPPED': 'IS_PO_OVERSHIPPED', 'IS_PO_OVERRECEIVED': 'IS_PO_OVERRECEIVED', 'IS_PO_OVERINVOICED': 'IS_PO_OVERINVOICED', 'IS_PO_HAS_SUBSTITUTED_INVOICE_ITEMS': 'IS_PO_HAS_SUBSTITUTED_INVOICE_ITEMS', 'IS_PO_HAS_CANCELLED_RECEIPT_ITEMS': 'IS_PO_HAS_CANCELLED_RECEIPT_ITEMS', 'IS_PO_HAS_CANCELLED_ITEMS': 'IS_PO_HAS_CANCELLED_ITEMS', 'IS_PO_HAS_RECEIPTS': 'IS_PO_HAS_RECEIPTS', 'IS_PO_HAS_RETURNS': 'IS_PO_HAS_RETURNS', 'IS_PO_HAS_INVOICES': 'IS_PO_HAS_INVOICES', 'UNIT_PRICE_USD': 'UNIT_PRICE_USD', 'EXTENDED_PRICE_USD': 'EXTENDED_PRICE_USD', 'CONTRACT_UNIT_PRICE_USD': 'CONTRACT_UNIT_PRICE_USD', 'SUPPLIER_RANK': 'SUPPLIER_RANK', 'DIVERSIFIED_SUPPLIER': 'DIVERSIFIED_SUPPLIER', 'LIST_PRICE_SET_ID': 'LIST_PRICE_SET_ID', 'LIST_PRICE_SET_NAME': 'LIST_PRICE_SET_NAME', 'LIST_PRICE': 'LIST_PRICE', 'LIST_PRICE_SET_VERSION': 'LIST_PRICE_SET_VERSION', 'PREVIOUS_LIST_PRICE': 'PREVIOUS_LIST_PRICE', 'PREVIOUS_LIST_PRICE_SET_VERSION': 'PREVIOUS_LIST_PRICE_SET_VERSION', 'PO_TYPE_ENUM': 'PO_TYPE_ENUM', 'PO_TYPE': 'PO_TYPE', 'PO_RECEIPT_STATUS_ENUM': 'PO_RECEIPT_STATUS_ENUM', 'PO_RECEIPT_STATUS': 'PO_RECEIPT_STATUS', 'PO_INVOICE_STATUS_ENUM': 'PO_INVOICE_STATUS_ENUM', 'PO_INVOICE_STATUS': 'PO_INVOICE_STATUS', 'PO_WORKFLOW_STATUS_ENUM': 'PO_WORKFLOW_STATUS_ENUM', 'PO_WORKFLOW_STATUS': 'PO_WORKFLOW_STATUS', 'PO_MATCH_STATUS_ENUM': 'PO_MATCH_STATUS_ENUM', 'PO_MATCH_STATUS': 'PO_MATCH_STATUS', 'UNIT_PRICE_SOURCE_ENUM': 'UNIT_PRICE_SOURCE_ENUM', 'UNIT_PRICE_SOURCE': 'UNIT_PRICE_SOURCE', 'PO_LINE_MATCH_STATUS_ENUM': 'PO_LINE_MATCH_STATUS_ENUM', 'PO_LINE_MATCH_STATUS': 'PO_LINE_MATCH_STATUS', 'CONTRACT_UNIT_PRICE_BUSINESS': 'CONTRACT_UNIT_PRICE_BUSINESS', 'CONTRACT_UNIT_PRICE_BUSINESS_CURRENCY': 'CONTRACT_UNIT_PRICE_BUSINESS_CURRENCY', 'CONTRACT_UNIT_PRICE_BUSINESS_EXCHANGE_RATE': 'CONTRACT_UNIT_PRICE_BUSINESS_EXCHANGE_RATE', 'EXTENDED_PRICE_BUSINESS': 'EXTENDED_PRICE_BUSINESS', 'EXTENDED_PRICE_BUSINESS_CURRENCY': 'EXTENDED_PRICE_BUSINESS_CURRENCY', 'EXTENDED_PRICE_BUSINESS_EXCHANGE_RATE': 'EXTENDED_PRICE_BUSINESS_EXCHANGE_RATE', 'UNIT_PRICE_BUSINESS': 'UNIT_PRICE_BUSINESS', 'UNIT_PRICE_BUSINESS_CURRENCY': 'UNIT_PRICE_BUSINESS_CURRENCY', 'UNIT_PRICE_BUSINESS_EXCHANGE_RATE': 'UNIT_PRICE_BUSINESS_EXCHANGE_RATE', 'IS_PO_LINE_HAS_SHIPPED_ITEMS': 'IS_PO_LINE_HAS_SHIPPED_ITEMS', 'IS_PO_LINE_HAS_RECEIPTS': 'IS_PO_LINE_HAS_RECEIPTS', 'PO_LINE_RECEIPT_STATUS_ENUM': 'PO_LINE_RECEIPT_STATUS_ENUM', 'PO_LINE_RECEIPT_STATUS': 'PO_LINE_RECEIPT_STATUS', 'PO_LINE_SHIPMENT_STATUS_ENUM': 'PO_LINE_SHIPMENT_STATUS_ENUM', 'PO_LINE_SHIPMENT_STATUS': 'PO_LINE_SHIPMENT_STATUS', 'RECEIPT_MAX_UNIT_PRICE': 'RECEIPT_MAX_UNIT_PRICE', 'RECEIPT_MIN_UNIT_PRICE': 'RECEIPT_MIN_UNIT_PRICE', 'IS_PO_LINE_WITH_CANCELLED_RECEIPT_ITEMS': 'IS_PO_LINE_WITH_CANCELLED_RECEIPT_ITEMS', 'IS_PO_LINE_MATCHING_NEEDS_RECEIPT': 'IS_PO_LINE_MATCHING_NEEDS_RECEIPT', 'IS_PO_LINE_MATCHING_ONLY_NEEDS_RECEIPT': 'IS_PO_LINE_MATCHING_ONLY_NEEDS_RECEIPT', 'IS_PO_LINE_OVERSHIPPED': 'IS_PO_LINE_OVERSHIPPED', 'RECEIPT_MAX_UNIT_PRICE_USD': 'RECEIPT_MAX_UNIT_PRICE_USD', 'RECEIPT_MIN_UNIT_PRICE_USD': 'RECEIPT_MIN_UNIT_PRICE_USD', 'SHIP_TO_ADDRESS_ID': 'SHIP_TO_ADDRESS_ID', 'BILL_TO_ADDRESS_ID': 'BILL_TO_ADDRESS_ID', 'COMMODITY_CODE_KEY': 'COMMODITY_CODE_KEY', 'REQUESTED_DELIVERY_DATE': 'REQUESTED_DELIVERY_DATE', 'DELIVERY_DATE_TYPE': 'DELIVERY_DATE_TYPE', 'DELIVERY_LEAD_TIME_DAYS': 'DELIVERY_LEAD_TIME_DAYS', 'FORM_ID': 'FORM_ID', 'FORMREQUEST_ID': 'FORMREQUEST_ID', 'GRAND_TOTAL_AMOUNT': 'GRAND_TOTAL_AMOUNT', 'GRAND_TOTAL_DOCUMENT_AMOUNT': 'GRAND_TOTAL_DOCUMENT_AMOUNT', 'GRAND_TOTAL_DOCUMENT_CURRENCY': 'GRAND_TOTAL_DOCUMENT_CURRENCY', 'GRAND_TOTAL_DOCUMENT_EXCHANGE_RATE': 'GRAND_TOTAL_DOCUMENT_EXCHANGE_RATE', 'GRAND_TOTAL_USD': 'GRAND_TOTAL_USD', 'ELT_TS': 'ELT_TS', 'FULFILLMENT_CENTER_KEY': 'FULFILLMENT_CENTER_KEY', 'CATEGORY_LEVEL_1_NAME': 'CATEGORY_LEVEL_1_NAME', 'CATEGORY_LEVEL_1_UNSPSC': 'CATEGORY_LEVEL_1_UNSPSC', 'CATEGORY_LEVEL_2_NAME': 'CATEGORY_LEVEL_2_NAME', 'CATEGORY_LEVEL_2_UNSPSC': 'CATEGORY_LEVEL_2_UNSPSC'}, 'JI_PURCHASE_ORDER': {'ORG_ID': 'ORG_ID', 'PO_ID': 'PO_ID', 'DEPT_KEY': 'DEPT_KEY', 'PO_NUMBER': 'PO_NUMBER', 'EXTERNAL_PO_ID': 'EXTERNAL_PO_ID', 'PO_LAST_REVISION_NUMBER': 'PO_LAST_REVISION_NUMBER', 'IS_PO_HAS_REJECTED_ITEMS': 'IS_PO_HAS_REJECTED_ITEMS', 'IS_PO_HAS_CREDITS': 'IS_PO_HAS_CREDITS', 'IS_PO_OVERSHIPPED': 'IS_PO_OVERSHIPPED', 'IS_PO_OVERRECEIVED': 'IS_PO_OVERRECEIVED', 'IS_PO_OVERINVOICED': 'IS_PO_OVERINVOICED', 'IS_PO_HAS_SUBSTITUTED_INVOICE_ITEMS': 'IS_PO_HAS_SUBSTITUTED_INVOICE_ITEMS', 'IS_PO_HAS_CANCELLED_RECEIPT_ITEMS': 'IS_PO_HAS_CANCELLED_RECEIPT_ITEMS', 'IS_PO_HAS_CANCELLED_ITEMS': 'IS_PO_HAS_CANCELLED_ITEMS', 'IS_PO_HAS_RECEIPTS': 'IS_PO_HAS_RECEIPTS', 'IS_PO_HAS_RETURNS': 'IS_PO_HAS_RETURNS', 'IS_PO_HAS_INVOICES': 'IS_PO_HAS_INVOICES', 'IS_PO_FORCE_MATCHED': 'IS_PO_FORCE_MATCHED', 'CREATED_TS': 'CREATED_TS', 'DISTRIBUTION_TS': 'DISTRIBUTION_TS', 'EXPORT_TS': 'EXPORT_TS', 'LASTREVISION_TS': 'LASTREVISION_TS', 'ORIGINALREVISION_TS': 'ORIGINALREVISION_TS', 'WORKFLOWCOMPLETED_TS': 'WORKFLOWCOMPLETED_TS', 'ACCOUNTING_DATE': 'ACCOUNTING_DATE', 'USER_OWNER_KEY': 'USER_OWNER_KEY', 'USER_SUBMITTER_KEY': 'USER_SUBMITTER_KEY', 'SUPPLIER_KEY': 'SUPPLIER_KEY', 'GRAND_TOTAL': 'GRAND_TOTAL', 'GRAND_TOTAL_USD': 'GRAND_TOTAL_USD', 'PO_TYPE_ENUM': 'PO_TYPE_ENUM', 'PO_TYPE': 'PO_TYPE', 'PO_RECEIPT_STATUS_ENUM': 'PO_RECEIPT_STATUS_ENUM', 'PO_RECEIPT_STATUS': 'PO_RECEIPT_STATUS', 'PO_INVOICE_STATUS_ENUM': 'PO_INVOICE_STATUS_ENUM', 'PO_INVOICE_STATUS': 'PO_INVOICE_STATUS', 'PO_WORKFLOW_STATUS_ENUM': 'PO_WORKFLOW_STATUS_ENUM', 'PO_WORKFLOW_STATUS': 'PO_WORKFLOW_STATUS', 'PO_AP_STATUS_ENUM': 'PO_AP_STATUS_ENUM', 'PO_AP_STATUS': 'PO_AP_STATUS', 'PO_MATCH_STATUS_ENUM': 'PO_MATCH_STATUS_ENUM', 'PO_MATCH_STATUS': 'PO_MATCH_STATUS', 'GRAND_TOTAL_BUSINESS': 'GRAND_TOTAL_BUSINESS', 'GRAND_TOTAL_BUSINESS_CURRENCY': 'GRAND_TOTAL_BUSINESS_CURRENCY', 'GRAND_TOTAL_BUSINESS_EXCHANGE_RATE': 'GRAND_TOTAL_BUSINESS_EXCHANGE_RATE', 'ELT_TS': 'ELT_TS'}}}]

In [50]:
unioned_tables_node = processor.run_union_operations(
    processor.ctx.table_store.get_all_tables_in_db_schema(BEFORE_UNION_DB_SCHEMA),
    union_operations,
    # [union_operations_node],
)
unioned_tables = unioned_tables_node.computation_output
print(unioned_tables)

{'Union_1': <processor.table.representation.impl.df_table.DFTable object at 0x7f18bc60ef30>}


/zp_more/project_data/pneuma/processor/src/processor/table/representation/impl/df_table.py:119: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = concat(dfs, ignore_index=True)
/zp_more/project_data/pneuma/processor/src/processor/table/representation/impl/df_table.py:119: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = concat(dfs, ignore_index=True)
/zp_more/project_data/pneuma/processor/src/processor/table/representation/impl/df_table.py:119: FutureWarning: The behavior of

In [ ]:
# import pandas as pd
# import numpy as np

# def clean_duplicate_ids(df: pd.DataFrame, id_col='ID') -> pd.DataFrame:
#     df = df.copy()
#     # 1) Record original dtypes
#     orig_dtypes = df.dtypes.copy()

#     non_id_cols = [col for col in df.columns if col != id_col]
#     # 2) Ensure df itself uses pandas’ nullable types
#     for col in non_id_cols:
#         dt = orig_dtypes[col]
#         if dt.kind in ('i',):        # integer
#             df[col] = df[col].astype("Int64")
#         elif dt.kind in ('f',):      # float
#             df[col] = df[col].astype("Float64")
#         elif dt == object:           # string/object
#             df[col] = df[col].astype("string")

#     grouped = df.groupby(id_col, dropna=False)
#     cleaned_rows = []

#     for id_val, group in grouped:
#         if len(group) == 1:
#             cleaned_rows.append(group.iloc[0].to_dict())
#             continue

#         values_dict = {}
#         misalign_count = total_checks = 0

#         for col in non_id_cols:
#             non_null_vals = group[col].dropna().unique()
#             values_dict[col] = non_null_vals.tolist()
#             if len(non_null_vals) > 1:
#                 misalign_count += 1
#             if len(non_null_vals) >= 1:
#                 total_checks += 1

#         if total_checks > 0 and misalign_count == total_checks:
#             cleaned_rows.extend(group.to_dict(orient="records"))
#             continue

#         max_len = max(len(v) for v in values_dict.values())
#         if max_len <= 1:
#             combined = {col: (vals[0] if vals else np.nan)
#                         for col, vals in values_dict.items()}
#             combined[id_col] = id_val
#             cleaned_rows.append(combined)
#         else:
#             for i in range(max_len):
#                 row = {}
#                 for col, vals in values_dict.items():
#                     if len(vals) == 1:
#                         row[col] = vals[0]
#                     elif i < len(vals):
#                         row[col] = vals[i]
#                     else:
#                         row[col] = np.nan
#                 row[id_col] = id_val
#                 cleaned_rows.append(row)

#     # 3) Build the result and restore nullable dtypes
#     result = pd.DataFrame(cleaned_rows)[df.columns].reset_index(drop=True)
#     for col in result.columns:
#         orig_dt = orig_dtypes[col]
#         if orig_dt.kind in ('i',):
#             result[col] = result[col].astype("Int64")
#         elif orig_dt.kind in ('f',):
#             result[col] = result[col].astype("Float64")
#         elif orig_dt == object:
#             result[col] = result[col].astype("string")

#     return result
# y = unioned_tables['Union_1'].data
# # y = y[y['ASN_ID'] == 17920751]
# z = y.T.drop_duplicates().T
# z_cleaned = clean_duplicate_ids(z, 'PO_ID')
# unioned_tables['Union_1'].data = z_cleaned

In [56]:
y = unioned_tables['Union_1'].data
y = y.T.drop_duplicates().T
unioned_tables['Union_1'].data = y

In [57]:
# Save to DB
try:
    processor.ctx.table_store.delete_db_schema(DB_SCHEMA_AFTER_UNION)
    processor.ctx.table_store.create_db_schema(DB_SCHEMA_AFTER_UNION)
except:
    pass
for table_id, table in unioned_tables.items():
    processor.ctx.table_store.add_table(DB_SCHEMA_AFTER_UNION, table_id, table, True, False)
processor.ctx.table_store.checkpoint()

In [25]:
join_desc_sys_prompt = """You are given a table that was formed using multiple tables. Each of these source tables have its own description.

Your goal is to describe briefly what this table represents (roughly as long as the individual source table descriptions)."""

> 37s

In [26]:
for union_op in union_operations:
    tables = union_op["Tables"]
    union_table = union_op["Output Table ID"]
    concatenated_descriptions = ""
    for table in tables:
        table_desc = table_descriptions[table]
        concatenated_descriptions += f"- {table_desc}\n"
    concatenated_descriptions = concatenated_descriptions.strip()
    msg = [
        {'role': 'system', 'content': join_desc_sys_prompt},
        {'role': 'user', 'content': f"- Table: ```{processor.ctx.table_store.get_table(DB_SCHEMA_AFTER_UNION, union_table).get_representation(3)}```\n- Individual descriptions: ```{concatenated_descriptions}```"},
    ]
    llm_output = processor.ctx.llm.chat(msg)
    table_descriptions[union_table] = llm_output

In [27]:
table_descriptions

{'JI_DEPARTMENT': 'Provides details on contract related departments.',
 'JI_DEPARTMENT_BUSINESS_UNIT': 'Provides department details for EPRO and Form Request documents.',
 'JI_FULFILLMENT_CENTER_CORREL_BUSINESS_UNIT': 'Provides details on the business units that are assigned to a fulfillment center.',
 'JI_FULFILLMENT_CENTER': 'Provides details on the fulfillment center that is associated with a supplier.',
 'JI_CORREL_ORIGINAL_REQ_LINE_TO_SOURCING_EVENT': 'Provides the association between an original requisition and a sourcing event.',
 'JI_COMMODITY_CODE': 'Provides commodity code details on EPRO related documents.',
 'JI_BUSINESS_UNIT': 'Provides details on the business unit',
 'JI_ADDRESS': 'Provides details on addresses.',
 'JI_FULFILLMENT_CENTER_TERMS_CONDITIONS': 'Provides details on purchase order terms and conditions as well as payment terms.',
 'JI_PAYMENT': 'Provides details on paid and settled dates.',
 'JI_PAYMENT_EVENTS': 'Provides details on the payment events.',
 'JI_JA

## 2d. Table Join

In [ ]:
# Skipped

## 2e. Save to DB

In [58]:
processor.ctx.table_store.delete_db_schema(DB_SCHEMA_FINAL)
processor.ctx.table_store.create_db_schema(DB_SCHEMA_FINAL)
base_table = processor.ctx.table_store.get_table(DB_SCHEMA_AFTER_UNION, 'Union_1')
processor.ctx.table_store.add_table(
    DB_SCHEMA_FINAL,
    "base_table",
    base_table,
    True,
    True,
)

# Step 3: Base Table Reducer

## 3a. Define Constants

### Prompts, target schema, SQL

In [59]:
target_schema = {
    "PO_ID": {"description": "Purchase Order ID", "type": "VARCHAR"},
    "Supplier": {"description": "Name of the supplier", "type": "VARCHAR"},
    "Country_of_Supplier": {
        "description": "Country where the supplier is located",
        "type": "VARCHAR",
    },
    "Import_Tariff": {
        "description": "Import tariff percentage applied to goods from the supplier",
        "type": "FLOAT",
    },
    "Monetary_Value_USD": {
        "description": "Monetary value of the order in USD",
        "type": "FLOAT",
    },
}
sql_query = "SELECT (Import_Tariff / 100 * Monetary_Value_USD) AS Tariff_Difference FROM target_table WHERE PO_ID = 40411289"

In [94]:
base_table_reducer_prompts = {
    "python_column_extractor": """You are an experienced data scientist. Given a table represented as a Pandas DataFrame, your task is to write a Python function named generate_column with no arguments except for the dataframe itself. The function must return a list representing the values of a new column, one for each row in the DataFrame.

You are also given a natural language question, which provides context for the computation, such as filtering criteria (e.g., specific PO ID, date, or status) or numeric rules (e.g., tariffs, thresholds). Always refer closely to the filtering criteria explicitly mentioned in the question or SQL — do not invent new conditions based on assumptions.

Guidelines:
- Only use columns that are present in the input DataFrame.
- Always convert column data types before applying logic to avoid errors.
- Handle missing (null) values carefully.
- Use regex for string equality checks where applicable.
- If a subset of rows is targeted, assign appropriate values only to them and ensure the output list length matches the number of rows. Set default (e.g., `None`) for other rows.
- Avoid speculative logic. For example, if a specific PO ID is provided, do not try to filter by country or supplier unless explicitly needed for the computation. Rely on retrieved context for numeric constants like tariffs or thresholds — do not invent values.

Output only the function definition for `generate_column`, with no additional text or explanation.""",
    "column_projection": """You are a helpful data scientist.

You will be provided with:
- A source table called SRC that is represented by its schema and some sample rows.
- A target schema that we will transform the schema of source table into it in a step-by-step manner.
- A question that we want to answer, which was used to form the target schema.
- A column from the target schema as the current target column.

Your goal is to determine whether to select a certain column from SRC or extract information from certain column(s) from SRC to form the target column (even as simple as adding time delta to each row).
Extract_column can relies on external tools such as Python code interpreter, SQL processor, or LLM.

When using extract_column, always include all columns from SRC that are required to perform the extraction, even if their role seems minor or indirect. These can include helper columns (e.g., timestamps for computing durations, ZIP codes for locations, etc.).

The output format for selecting a certain column:
{
    "operation": "select_column",
    "description": "Select SRC.Restaurant ID."
    "columns_involved": ["Restaurant ID"],
}

While for extracting information from certain column(s):
{
    "operation": "extract_column",
    "description": "Find the country based on SRC.City and SRC.`ZIP Code`."
    "columns_involved": ["City", "ZIP Code"],
}

NOTE: consider the question very carefully when determining how to get the current target column, as it may cue what the target schema means.

Output your result strictly as a Python dictionary, without any extra formatting, explanations, or text. The output must be directly parseable as a Python dictionary.""",
    "extract_mode": """You are a data scientist working with structured tables.

You will be given:
- A table (schema and sample rows).
- A new column to generate, which is needed to answer a question.

Your job is to decide:
1. Should the values of the new column be extracted row-by-row using language reasoning by LLM?
2. Or, can the values be generated using a single Python function that processes the other column(s)?

Output one of:
- 'rowwise_extraction'
- 'python_code'

Output a JSON object directly without any quotes, explanations, or formatting with the following format:
{
    "type": "python_code/rowwise_extraction"
    "explanation": "Explanation of the computations that need to be done to produce the column."
}

Carefully interpret what the new column expects based on the given question, and prioritize python_code, which utilizes Pandas DataFrame, unless LLM is strictly necessary.""",
    "extract_col": """You are a helpful and knowledgeable data scientist.

You will be provided with:
- A table represented by its schema and rows.
- A column to be added to this table whose values depend on the other columns in the table.

Your goal is to determine the values of the new column for all rows. Ensure you consider **all provided columns together** rather than relying on a single column. For example, a city name may exist in multiple locations, but when paired with its corresponding province or county, ambiguity is reduced.

Output your result strictly as a Python list representing the new column values for all rows, without any extra formatting, explanations, or text. The output must be directly parseable as a Python list.""",
    "reduce_row": """You are a helpful and knowledgeable data scientist.

You will be provided with:
- A user's question.
- A table that combines multiple source of information to answer the question.

Your goal is to produce a SQL code (DuckDB) containing predicates to reduce the rows of target_table. In other words, you need to eliminate irrelevant rows.
However, you should do so carefully, especially when comparing equality (e.g., use "LOWER(name) LIKE LOWER('%john%')").

Extra note:
- DuckDB already understands date columns, so you do not need to wrap such columns with `DATE()`.

Output your result strictly as a SQL code (DuckDB) without any extra formatting, explanations, or text. The output must be directly parseable as a SQL code.""",
}

In [95]:
column_extraction_new_prompt = """You are a helpful data scientist.

You will be provided with:
- A source table called SRC that is represented by its schema and some sample rows.
- A target schema, in which a SQL statement will be executed over to answer a user's question.
- A single current target column, which is a key-value pair from the target schema (e.g., "Delayed Ship Date": {"description": "Actual ship date after the 3-day delay", "type": "DATE"}).

Your task is to decide whether to:
1. Select a column from SRC directly (if it maps cleanly to the target column), or
2. Extract or compute the target column using one or more columns from SRC (e.g., adding a time delta, parsing a field, applying a condition).

When deciding this, you must:
- Carefully analyze both the name and explanation of the current target column.
- Consider how the current column helps the execution of the SQL statement.
- Determine what transformation or logic is required to form this column from the source data.

When using extract_column, always include all columns from SRC that are required to perform the extraction, including:
- Main columns used in transformation.
- Helper/reference columns (e.g., timestamps, ZIP codes).
- Any columns used for conditional logic or filters (e.g., shipment method, status).

Be very careful about column names of SRC; do not exclude underscore symbols or whitespaces in the column names of SRC, as these result in errors.

Output format:

If you can directly select a column:
```python
{
    "operation": "select_column",
    "description": "Select SRC.Restaurant ID.",
    "columns_involved": ["Restaurant ID"],
}
```

If the target column must be derived:
```python
{
    "operation": "extract_column",
    "description": "Find the country based on SRC.City and SRC.`ZIP Code`.",
    "columns_involved": ["City", "ZIP Code"],
}
```

Your output must be a single valid Python dictionary. Do not add any extra text, markdown, or formatting."""

In [62]:
web_search_prompt = """You are helping a system determine whether it needs to search the web to compute a new column in a structured table.

You will be given:
- A table with selected columns.
- A SQL statement referencing the table.
- A question that will be answered using the SQL statement over the table.
- A new column to generate.
- A description of what the new column is supposed to represent.

Output a **raw JSON object** (no quotes, explanations, or extra formatting) in the following format:
{
    "need_web_search": "yes" or "no",
    "web_search_query": "..."  (leave blank if no)
}

Guidelines:
- Return "yes" if computing the new column requires up-to-date or external information not derivable from the existing table (e.g., stock prices, current events, real-world locations, latest tariffs, or official entity names).
- Return "no" if the data can be derived from the existing table or through logical inference alone.
- When "yes", create a query that would help find the necessary information via a search engine. Use relevant information from the question (e.g., minimum import tariffs set by the U.S. government)."""


In [80]:
extract_mode = """You are helping determine how to generate a new column in a table based on a natural language question. You will be given:
- A natural language question
- A table schema
- Sample rows from the table
- Additional retrieved context if available (e.g., web snippets or background knowledge)
- The SQL query being executed or referenced

Your task is to decide how to generate the new column. The goal is to use available information efficiently, even if the question or SQL only applies to a subset of rows (e.g., a specific PO ID, country, or event).

You must choose one of these options:
- 'rowwise_extraction': if each row must be analyzed using natural language reasoning (e.g., free-text fields or inferring states)
- 'python_code': if the column can be computed using known logic or numeric constants from the question, SQL statement, or retrieved context

Output a **raw JSON object** (no quotes, explanations, or extra formatting) in the following format:
{
    "type": "python_code" or "rowwise_extraction",
    "explanation": "Explain the reasoning or computation needed to generate the new column. Be thorough: include all entity details mentioned in the SQL statement. For instance, if the query refers to 'Amazon Same Day Service', do not generalize to just 'Amazon'. Additionally, if the question mentiones two filtering criteria (e.g., country and ID), but ID is sufficient to identify the target rows, then just use it to make the computation cheaper."
}

If a relevant constant (e.g., 10% tariff) is mentioned in the question or retrieved context, prefer 'python_code' and assume default or blank values for rows not matching the question's or SQL statement’s conditions.

Always consider the SQL statement and retrieved context when deciding, especially if they provide concrete logic or numeric values that can guide computation. Avoid defaulting to row-by-row reasoning if a simple rule can be applied."""


In [81]:
extract_col = """You are a helpful and knowledgeable data scientist.

You will be provided with:
- A table, represented by its schema and rows.
- A new column to be added, whose values depend on the existing columns.
- A description of how to transform the existing values into the new column values for all rows.

Your task is to generate the values for the new column, considering **all relevant columns together**, not in isolation. For example, a city name may appear in multiple regions, but when combined with the corresponding province or county, ambiguity can be resolved.

Output your answer as a **valid Python list** representing the new column values for all rows—**no extra text, formatting, or explanation**. The result must be directly parseable as a Python list."""


### Functions

In [96]:
from processor.conductor_state import ConductorState
from processor.table.representation.abstract_table import AbstractTable
from processor.utils.string_processor import clean_code_string, parse_code_string

import pandas as pd
import bm25s
import Stemmer

In [97]:
def web_search(
    question: str,
    index_path: str,
    k: int = 3,
) -> list[str]:
    retriever = bm25s.BM25.load(index_path, load_corpus=True)
    stemmer = Stemmer.Stemmer("english")

    query_tokens = bm25s.tokenize(question, stemmer=stemmer, show_progress=False)
    results, _ = retriever.retrieve(query_tokens, k=k, show_progress=False)

    search_results: list[str] = []
    for result_idx, result in enumerate(results[0]):
        search_results.append(result['text'])
    return search_results

In [98]:
import tqdm
def extract_column(
    ctx: ConductorState,
    sql_script: str,
    question: str,
    op_description: str,
    base_table: AbstractTable,
    columns_involved: list[str],
    target_column: str,
    row_batch=10,
    num_rows = 3,
    input_nodes: list = [],
):
    ctx.logger.info(f"===> Operation extract_column")
    sql_script = "SELECT "
    for col in columns_involved:
        if col in base_table.get_schema():
            sql_script += f'"{col}", '
    sql_script = sql_script[:-2] + " FROM base_table;"

    ctx.logger.info(f"===> sql_script: {sql_script}")
    columns_involved_table = ctx.table_store.execute_sql_query(
        sql_script,
        {
            "base_table": base_table,
        },
    )
    unique_columns_involved_table = columns_involved_table.drop_duplicates()

    # Get extra context using web search (if necessary)
    web_search_msg = [
        {
            "role": "system",
            "content": web_search_prompt,
        },
        {
            "role": "user",
            "content": f"- Table: ```{unique_columns_involved_table.get_representation(num_rows)}```\n- Target column: ```{target_column}```\n- SQL statement:\n```{sql_script}```\n- Description: ```{op_description}```\n- Question: ```{question}```",
        },
    ]
    web_search_decision = ctx.llm.chat(web_search_msg).strip()
    ctx.logger.info(f"===> Web search decision: {web_search_decision}")
    search_decision = parse_code_string(web_search_decision)

    web_context = ""
    if search_decision["need_web_search"] == "yes":
        query = search_decision["web_search_query"]
        search_snippets = web_search(query, "search_engine/indices/demo-index", 5)
        web_context = f"\n- Retrieved Web Snippets: ```{search_snippets}```"
    
    ctx.logger.info(f"===> Web search results: {web_context}")

    # Ask LLM whether to use row-wise extraction or Python code
    msg = [
        {
            "role": "system",
            "content": extract_mode,
        },
        {
            "role": "user",
            "content": f"- Table: ```{unique_columns_involved_table.get_representation(num_rows)}```\n- Overall Schema: ```{list(base_table.get_schema())}```\n- New Column: ```{target_column}```\n- SQL statement: ```{sql_script}```\n- Description: ```{op_description}```\n- Question: ```{question}```",
        },
    ]

    if len(web_context) > 0:
        msg[-1]["content"] += f"\n- Extra context (trustable AND relevant to the question): ```{web_context}```"

    output = ctx.llm.chat(msg).strip()
    ctx.logger.info(f"OUTPUT: {output}")
    extraction_mode: dict[str,str] = parse_code_string(output)
    ctx.logger.info(f"===> extraction_mode: {extraction_mode}")
    actual_values: list[str] = []
    if extraction_mode['type'] == "python_code":
        # Generate code from the LLM
        code_gen_msg = [
            {
                "role": "system",
                "content": base_table_reducer_prompts["python_column_extractor"],
            },
            {
                "role": "user",
                "content": f"- Table: ```{unique_columns_involved_table.get_representation(num_rows)}```\n- Target column: ```{target_column}```\n- SQL statement:\n```{sql_script}```\n- Computation to do: ```{extraction_mode['explanation']}```\n- Question: ```{question}```",
            },
        ]
        code_str = ctx.llm.chat(code_gen_msg)
        ctx.logger.info(f"Python code to extract: {code_str}")
        code_str = clean_code_string(code_str)
        exec_globals = {"pd": pd}
        exec(code_str, exec_globals)
        generated_func = exec_globals.get("generate_column")

        if not generated_func:
            raise ValueError("LLM did not return a valid 'generate_column' function.")
        actual_values = generated_func(columns_involved_table.get_data())
    elif extraction_mode['type'] == "rowwise_extraction":
        rows: list[tuple[int, int]] = []
        for i in range(0, len(unique_columns_involved_table), row_batch):
            rows.append((i, i + row_batch))
        rows[-1] = (rows[-1][0], len(unique_columns_involved_table))

        new_col_values: list[str] = []
        for row in tqdm(rows, desc="Processing column extraction"):
            msg = [
                {
                    "role": "system",
                    "content": extract_col,
                },
                {
                    "role": "user",
                    "content": f"- What to do: ```{extraction_mode['explanation']}```\n- Table ({row[1]-row[0]} rows): ```{unique_columns_involved_table.get_representation(row[1]-row[0], None, True, row)}```\n- Overall Schema: {list(base_table.get_schema())}\n- New Column: `{target_column}`",
                },
            ]
            extracted_values = ctx.llm.chat(msg)
            new_col_values.extend(parse_code_string(extracted_values))

        results_cache: dict[str, str] = dict()
        columns = unique_columns_involved_table.get_schema()
        for idx, row in unique_columns_involved_table.iterrows():
            vals = []
            for col in columns:
                vals.append(row[col])
            key = "_SEP_".join(vals)
            results_cache[key] = new_col_values[idx]

        for idx, row in columns_involved_table.iterrows():
            vals = []
            for col in columns:
                vals.append(row[col])
            key = "_SEP_".join(vals)
            actual_values.append(results_cache[key])

    return ctx.computation_graph.create_node(
        "Extraced column values from existing columns in the base table.",
        actual_values,
        input_nodes,
    )

In [99]:
def compute_target_table(
    ctx: ConductorState,
    sql_script: str,
    question: str,
    base_table: AbstractTable,
    target_schema: dict[str, str],
    num_rows=3,
    input_nodes = [],
    ):
        """
        Projects `base_table`, specifically its schema, to the `target_schema`,
        resulting in `target_table`.
        """
        ctx.logger.info("Computing target table")
        target_table_cols: dict[str, list] = dict()
        extra_input_nodes: list = []
        for col in target_schema:
            if col == 'Country_of_Supplier':
                continue
            ctx.logger.info(f"=> Processing column {col}")
            msg = [
                {
                    "role": "system",
                    "content": column_extraction_new_prompt,
                },
                {
                    "role": "user",
                    "content": f"- Source table: ```{base_table.get_representation(num_rows, 42)}```\n- SQL statement: ```{question}```\n- Target Schema: ```{target_schema}```\n- Target Column: ```{col}: {target_schema[col]}```",
                },
            ]
            operation: dict[str, str] = parse_code_string(ctx.llm.chat(msg))
            ctx.logger.info(f"==> Operation: {operation}")
            if operation["operation"] == "select_column":
                try:
                    operation_node = ctx.computation_graph.create_node(
                        "Mapped a column directly.",
                        list(base_table[operation["columns_involved"][0]]),
                        input_nodes,
                    )
                except:
                    base_table_cols = base_table.get_schema()
                    operation_node = ctx.computation_graph.create_node(
                        "Mapped a column directly (no mapping actually; empty values).",
                        [None] * len(base_table[base_table_cols[0]]),
                        input_nodes,
                    )
            else:
                ctx.logger.info("WARNING: ENTERING EXTRACT_COLUMN")
                operation_node = extract_column(
                    ctx,
                    sql_script,
                    question,
                    operation["description"],
                    base_table,
                    operation["columns_involved"],
                    f"{col}: {target_schema[col]}",
                    10,
                    num_rows,
                    input_nodes,
                )
            extra_input_nodes.append(operation_node)
            target_table_cols[col] = operation_node.computation_output
        target_table = type(base_table).merge_columns(target_table_cols)
        return ctx.computation_graph.create_node(
            "Projected columns from base table to target table.",
            target_table,
            input_nodes + extra_input_nodes,
        )

## 3b. Target Table Producer

> 2m33s

In [100]:
target_table_node = compute_target_table(
    processor.ctx,
    sql_query,
    QUESTION,
    processor.ctx.table_store.get_table(DB_SCHEMA_FINAL, "base_table"),
    target_schema,
    2,
    # [join_operations_node],
)
target_table = target_table_node.computation_output
print(target_table)

[2025-05-06 07:55:04] INFO in 1680718374: Computing target table
[2025-05-06 07:55:04] INFO in 1680718374: => Processing column PO_ID
[2025-05-06 07:55:19] INFO in 1680718374: ==> Operation: {'operation': 'select_column', 'description': 'Select SRC.PO_ID.', 'columns_involved': ['PO_ID']}
[2025-05-06 07:55:19] INFO in 1680718374: => Processing column Supplier
[2025-05-06 07:55:34] INFO in 1680718374: ==> Operation: {'operation': 'select_column', 'description': 'Select SRC.Supplier_KEY.', 'columns_involved': ['Supplier_KEY']}
[2025-05-06 07:55:34] INFO in 1680718374: => Processing column Import_Tariff
[2025-05-06 07:55:56] INFO in 1680718374: ==> Operation: {'operation': 'extract_column', 'description': 'Extract the import tariff percentage for the supplier from Germany associated with PO ID 40411289.', 'columns_involved': ['PO_ID', 'SUPPLIER_KEY', 'IMPORT_TARIFF_PERCENTAGE']}
[2025-05-06 07:55:56] INFO in 1680718374: WARNING: ENTERING EXTRACT_COLUMN
[2025-05-06 07:55:56] INFO in 1056477

In [101]:
QUESTION

'A supplier from Germany, associated with PO ID 40411289, has just announced the application of minimum import tariffs set by the U.S. government on goods imported from Germany. Previously, there were no such tariffs. How much of a difference will this make in terms of monetary value (in USD)?'

In [104]:
processor.ctx.table_store.add_table(DB_SCHEMA_FINAL, "target_table", target_table, True, True)

### Predicate

In [ ]:
# NATURALIZATION
# import pandas as pd
# table = processor.ctx.table_store.get_table(DB_SCHEMA_FINAL, "target_table")
# table_data: pd.DataFrame = table.data
# for col in table_data.columns:
#     if table_data[col].dtype == 'object':
#         try:
#             converted = pd.to_datetime(table_data[col], errors='raise')
#             table_data[col] = converted
#         except (ValueError, TypeError):
#             pass  # Not a datetime column
# table.data = table_data
# processor.ctx.table_store.checkpoint()

/tmp/ipykernel_4133677/3314272484.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(table_data[col], errors='raise')


In [107]:
apply_pred_prompt = """You are an expert data engineer.

You are given:
- A column of values from a table (as a list).
- A SQL statement that will be executed on the table.

Your task is to transform the values in the column according to the SQL statement as follows:
- If the value is **not mentioned** in the SQL statement, keep it as-is.
- If the value **matches or partially matches** a value mentioned in the SQL WHERE clause, **replace the entire value** with the corresponding value from the SQL statement. Do not preserve parts of the original string.

For example, if the SQL contains `WHERE Service_Provider = 'Amazon Same Day Delivery'` and a column value is `'ASD - Amazon Same Day'`, transform it to `'Amazon Same Day Delivery'` (notice "ASD" is also removed).

Output a Python list containing the transformed values, with no extra explanations or formatting."""

In [108]:
from processor.conductor_state import ConductorState
from processor.table.representation.abstract_table import AbstractTable
from pandas.core.dtypes.common import is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype
from tqdm import tqdm

def standardize(
    ctx: ConductorState,
    target_table: AbstractTable,
    sql_statement: str,
):
    ctx.logger.info(f"Processing schema: {target_table.get_schema()}")
    for col_name in target_table.get_schema():
        ctx.logger.info(f"=> Processing col: {col_name}")

        col_values = target_table[col_name]

        is_categorical_col = (
            (is_categorical_dtype(col_values) or is_object_dtype(col_values))
            and not is_datetime64_any_dtype(col_values)
        )
        exists_in_sql = col_name.lower() in sql_statement.lower()

        if is_categorical_col and exists_in_sql:
            ctx.logger.info(f"==> {col_name} is categorical, proceeds...")

            unique_col_values = col_values.drop_duplicates().dropna()
            rows: list[tuple[int, int]] = []
            for i in range(0, len(unique_col_values), 5):
                rows.append((i, i + 5))
            rows[-1] = (rows[-1][0], len(unique_col_values))

            new_col_values_cache: dict[str, str] = dict()
            for row in tqdm(rows, desc="Standardizing col values"):
                curr_unique_col_values = list(unique_col_values[row[0]:row[1]])
                ctx.logger.info(f"===> curr_unique_col_values: {curr_unique_col_values}")
                msg = [
                    {
                        "role": "system",
                        "content": apply_pred_prompt,
                    },
                    {
                        "role": "user",
                        "content": f"- Column values ({row[1]-row[0]} rows): ```{curr_unique_col_values}```\n- SQL statement : ```{sql_statement}```",
                    },
                ]
                extracted_values = parse_code_string(ctx.llm.chat(msg))
                ctx.logger.info(f"===> extracted_values: {extracted_values}")
                for i in range(len(extracted_values)):
                    extracted_value = extracted_values[i]
                    original_value = curr_unique_col_values[i]
                    new_col_values_cache[original_value] = extracted_value
            ctx.logger.info(new_col_values_cache)
            # ctx.logger.info(f"==> CACHE: {new_col_values_cache}")
            new_col_values = []
            for val in col_values:
                try:
                    new_col_values.append(new_col_values_cache[val])
                except:
                    new_col_values.append(val)
            target_table[col_name] = new_col_values
    return target_table

In [109]:
std_target_table = standardize(
    processor.ctx,
    processor.ctx.table_store.get_table(DB_SCHEMA_FINAL, "target_table").copy(),
    sql_query,
)

[2025-05-06 08:00:09] INFO in 3168743303: Processing schema: ['PO_ID', 'Supplier', 'Import_Tariff', 'Monetary_Value_USD']
[2025-05-06 08:00:09] INFO in 3168743303: => Processing col: PO_ID
[2025-05-06 08:00:09] INFO in 3168743303: => Processing col: Supplier
[2025-05-06 08:00:09] INFO in 3168743303: => Processing col: Import_Tariff
[2025-05-06 08:00:09] INFO in 3168743303: => Processing col: Monetary_Value_USD


/tmp/ipykernel_1371396/3168743303.py:18: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  (is_categorical_dtype(col_values) or is_object_dtype(col_values))


In [113]:
processor.ctx.table_store.add_table(DB_SCHEMA_FINAL, "std_target_table", std_target_table, True, True)

In [114]:
def execute_sql_query(
    sql_query: str, tables_involved: dict[str, AbstractTable] = None
) -> AbstractTable:
    """
    [EXPERIMENTAL] Executes SQL query

    Args:
        sql_query (str): SQL query to execute
        tables_involved (list[AbstractTable]): OPTIONAL - Specify tables to query over (used by, e.g., PyTableStore)
    """
    import duckdb

    # Create an in-memory DuckDB connection
    conn = duckdb.connect(database=":memory:")

    # Make sure you pass a dictionary of DFTable
    if tables_involved is None or len(tables_involved) == 0:
        raise ValueError(
            "PyTableStore requires `tables_involved` to execute SQL queries."
        )

    if not isinstance(tables_involved[list(tables_involved.keys())[0]], DFTable):
        raise ValueError("Only Pandas DataFrame is supported for now.")

    # Register each DataFrame as a DuckDB view
    for table_id, table in tables_involved.items():
        df = table.get_data().copy()
        for col in df.columns:
            if df[col].dtype == "object":
                try:
                    df[col] = pd.to_datetime(df[col])
                except Exception:
                    pass  # Not datetime, ignore
        conn.register(table_id.lower(), df)

    # Run your SQL query (assumed lowercase)
    data = conn.execute(sql_query).fetchdf()
    return DFTable(data)

In [115]:
answer_table = execute_sql_query(
    sql_query,
    {
        "target_table": processor.ctx.table_store.get_table(DB_SCHEMA_FINAL, "std_target_table")
    }
)

In [116]:
answer_table.data

,Tariff_Difference
0,52.4


# Visualization

In [ ]:
from pyvis.network import Network
import networkx as nx

from processor.computation_graph import ComputationGraph


def interactive_network_pyvis(graph: ComputationGraph):
    G = nx.DiGraph()

    for node in graph.nodes:
        # label = f"{node.function_name}()\n{node.class_name or ''}\n{node.computation_description}"
        label = f"{node.function_name}()"
        G.add_node(node.id, label=label)

    for node in graph.nodes:
        for input_node in node.input_nodes:
            G.add_edge(input_node.id, node.id)

    net = Network(
        notebook=True,
        height="600px",
        width="100%",
        directed=True,
        cdn_resources="in_line",
    )
    net.from_nx(G)
    net.show("graph.html")  # Will now render inline in Jupyter

In [ ]:
interactive_network_pyvis(processor.ctx.computation_graph)

In [ ]:
# BACKUP OLD
enhanced_schemas = {
    "JI_ASN_CARRIER": [
        "ShippingNoticeID",
        "OrganizationID",
        "SHIPPER_CARRIER",
        "Shipment_Domain",
        "SHIPMENT_UNIQUE_ID",
        "ASN_Timestamp",
    ],
    "JI_PURCHASE_ORDER_LINE": [
        "ORGANIZATION_ID",
        "PURCHASE_ORDER_ID",
        "PO_LINE_ITEM_ID",
        "DEPARTMENT_KEY",
        "SUPPLIER_ID",
        "LINE_ITEM_KEY",
        "PURCHASE_ORDER_NUMBER",
        "CONTRACT_REFERENCE_ID",
        "CONTRACT_ID_NUMBER",
        "ORDER_QUANTITY",
        "TOTAL_LINE_COST",
        "ORDER_CREATION_TIMESTAMP",
        "DISTRIBUTION_SHIPMENT_TS",
        "EXPORT_TIMESTAMP",
        "LAST_REVISION_TIMESTAMP",
        "ORIGINAL_REVISION_TIMESTAMP",
        "WORKFLOW_COMPLETION_TS",
        "ACCOUNTING_DATE_TIMESTAMP",
        "USER_OWNER_ID",
        "USER_SUBMITTER_ID",
        "EXTERNAL_PURCHASE_ORDER_LINE_ID",
        "PO_LINE_NUMBER",
        "UNIT_PRICE_PER_ITEM",
        "CONTRACT_UNIT_PRICE_CONTRACTED",
        "SUPPLIER_ACCOUNTING_CODE",
        "SHIPMENT_METHOD",
        "IS_PO_LINE_AWARDED_BID_FLAG",
        "IS_PO_LINE_REJECTED_FLAG",
        "IS_PO_LINE_CANCELLED_FLAG",
        "IS_LINE_SENT_TO_SUPPLIER",
        "HAS_INVOICES",
        "IS_FORCE_MATCHED",
        "IS_PO_LINE_FORCED_MATCHED",
        "REQUISITION_REQUEST_ID",
        "REQUISITION_IDENTIFIER",
        "REQUISITION_LINE_NUMBER",
        "REQUISITION_LINE_ID",
        "REQUISITION_CREATION_TS",
        "PO_LATEST_REVISION_NUMBER",
        "HAS_REJECTED_RECEIPTS",
        "HAS_CREDITS",
        "IS_OVERSHIPPED",
        "IS_PO_LINE_EXCESS_RECEIPT",
        "IS_PO_LINE_OVERINVOICED",
        "HasSubstitutedInvoiceItems",
        "HAS_CANCELLED_RECEIPT_ITEMS",
        "IS_PO_LINE_HAS_CANCELLED_ITEMS",
        "HAS_RECEIVED_SHIPMENTS",
        "HAS_RETURN_RECEIPTS",
        "HAS_INVOICES",
        "UNIT_PRICE_IN_USD",
        "EXTENDED_PRICE_IN_USD",
        "CONTRACT_UNIT_PRICE_IN_USD",
        "SUPPLIER_RANKING",
        "IS_DIVERSE_SUPPLIER",
        "LIST_PRICE_SET_KEY",
        "LIST_PRICE_SET_DESCRIPTION",
        "CONTRACT_LIST_PRICE\n\nThis_name_better_reflects_that_the_value_in_this_column_is_likely_the_list_price_associated_with_the_contract_for_the_item,_which_is_specific_to_the_context_of_the_purchase_order_and_its_line_items._It_also_maintains_clarity_and_consistency_with_other_column_names_that_describe_pricing_aspects.",
        "LIST_PRICE_SET_VERSION_NUMBER",
        "PREVIOUS_LIST_PRICE_VERSION",
        "PREVIOUS_LIST_PRICE_SET_VERSION_NAME",
        "Purchase_Order_Type_Code",
        "Purchase_Order_Type",
        "RECEIPT_STATUS_ENUM",
        "RECEIPT_STATUS",
        "PO_Invoice_Status_Type",
        "INVOICE_STATUS",
        "PO_Workflow_Status_Type",
        "Purchase_Order_Workflow_Status",
        "PO_Match_Status_Enum",
        "PURCHASE_ORDER_MATCH_STATUS",
        "UNIT_PRICE_SOURCE_TYPE",
        "UNIT_PRICE_SOURCE_DESCRIPTION",
        "PO_LINE_MATCH_STATUS_DESCRIPTION",
        "PO_LINE_MATCHING_STATUS",
        "Contract_Unit_Price_Business",
        "CONTRACT_UNIT_PRICE_BUSINESS_CURRENCY_CODE",
        "CONTRACT_UNIT_PRICE_BUSINESS_EXCHANGE_RATE_DESCRIPTION",
        "EXTENDED_PRICE_BUSINESS_VALUE",
        "BUSINESS_EXTENDED_PRICE_CURRENCY",
        "EXTENDED_PRICE_BUSINESS_EXCHANGE_RATE_VALUE",
        "BUSINESS_UNIT_PRICE",
        "BUSINESS_UNIT_PRICE_CURRENCY",
        "EXCHANGE_RATE_BUSINESS_TO_USD",
        "HAS_SHIPPED",
        "HAS_RECEIVED_SHIPMENTS",
        "PO_LINE_RECEIPT_STATUS_DESCRIPTION\n\nThis_name_provides_clarity_about_the_nature_of_the_data_stored_in_the_column,_indicating_that_it_describes_the_status_of_receipts_for_each_PO_line.",
        "RECEIPT_STATUS_ENUM",
        "PO_LINE_SHIPMENT_STATUS_DESCRIPTION",
        "SHIPMENT_STATUS",
        "MAX_UNIT_PRICE_RECEIVED",
        "MINIMUM_RECEIPT_UNIT_PRICE",
        "HAS_CANCELLED_RECEIPT_ITEMS",
        "IS_PO_LINE_REQUIRES_RECEIPT_MATCHING",
        "IS_PO_LINE_MATCHING_REQUIRES_RECEIPT",
        "IS_PO_LINE_SHIPPED_IN_EXCESS",
        "MAX_RECEIPT_UNIT_PRICE_USD",
        "MIN_RECEIPT_UNIT_PRICE_USD",
        "SHIP_TO_ADDRESS_IDENTIFIER",
        "Bill_To_Address_ID",
        "COMMODITY_CODE_KEY",
        "REQUESTED_DELIVERY_DATE",
        "DELIVERY_DATE_CATEGORY",
        "DELIVERY_LEAD_TIME_IN_DAYS",
        "FORM_DOCUMENT_ID",
        "FORM_REQUEST_ID",
        "TOTAL_PURCHASE_ORDER_AMOUNT",
        "TOTAL_DOCUMENT_AMOUNT",
        "Document_Currency_Type",
        "Exchange_Rate_Grand_Total_Document",
        "Total_USD",
        "LAST_TRANSFORM_TS",
        "FULFILLMENT_CENTER_ID",
        "TOP_LEVEL_CATEGORY_NAME",
        "UNSPSC_Category_Level_1",
        "CATEGORY_LEVEL_2_DESCRIPTION",
        "UNSPSC_Category_Level_2",
    ],
    "JI_ORDER_ACK_LINE": [
        "Order_Acknowledgment_ID",
        "ShipmentLineIdentifier",
        "PURCHASE_ORDER_LINE_ID",
        "OrganizationID",
        "ORDER_QUANTITY",
        "EstimatedShippingDate",
        "Order_Status_Code",
        "Order_Ack_Status",
        "ACKNOWLEDGEMENT_NOTES",
        "LAST_UPDATED_TS",
    ],
    "JI_ASN_LINE": [
        "ASN_Line_ID",
        "ASN_LINE_ID",
        "PURCHASE_ORDER_LINE_ID",
        "Organization_ID",
        "Shipped_Quantity",
        "SHIPMENT_NOTES",
        "SHIPMENT_RECORD_TIMESTAMP",
    ],
    "JI_ASN": [
        "AdvancedShippingNoticeID",
        "Shipment_ID",
        "Originating_Organization_ID",
        "ScheduledShipmentDate",
        "DELIVERY_DATE",
        "SHIPMENT_COMMENTS",
        "LAST_UPDATE_TS",
    ],
}